# 05 · Encoders em português: BERTimbau e Albertina

Ajuste fino com duas fontes de rótulo **nos mesmos textos** do treino do RePro: humanos e LLM. A diferença de desempenho no teste mede quanto se perde trocando anotação humana por LLM.

Ambiente: **GPU T4** (~10 min por treino).

In [ ]:
# Configuração: no Colab, clona o repositório, instala o pacote `absa`
# e guarda cache do LLM, modelos e predições no Drive (sobrevive à queda da sessão)
import sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/vnevesl/pln-absa-olist.git 2>/dev/null || git -C pln-absa-olist pull -q
    %cd pln-absa-olist
    !pip -q install -e ".[transformers]"
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/pln-absa-olist/llm_cache /content/drive/MyDrive/pln-absa-olist/models /content/drive/MyDrive/pln-absa-olist/interim
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/llm_cache data/llm_cache
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/models models
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/interim data/interim

In [ ]:
import pandas as pd
from absa.config import INTERIM, MODELS
from absa.metrics import bootstrap_ci, per_aspect_report, tuple_f1
from absa.models.encoder import ENCODERS, EncoderAspectClassifier, TrainConfig
from absa.preprocess import clean_light
from absa.repro import load_repro
treino, val, teste, _ = load_repro()
print(len(treino), len(val), len(teste))

fontes = {"humano": treino}
arq = INTERIM / "repro_treino_rotulos_llm.csv"
if arq.exists():
    fontes["llm"] = pd.read_csv(arq)
X_val, X_teste = val.texto.map(clean_light).tolist(), teste.texto.map(clean_light).tolist()

In [ ]:
MODELS.mkdir(parents=True, exist_ok=True)
resultados = {}
for enc in ["bertimbau", "albertina"]:
    for fonte, rotulos in fontes.items():
        nome = f"{enc}_{fonte}"
        clf = EncoderAspectClassifier(TrainConfig(model_name=ENCODERS[enc], epochs=4, lr=3e-5, batch_size=32))
        # A seleção da melhor época usa a validação HUMANA nos dois casos
        clf.fit(rotulos.texto.map(clean_light).tolist(), rotulos, X_val, val, save_path=MODELS / f"{nome}.pt")
        clf.load(MODELS / f"{nome}.pt")
        pred = clf.predict(X_teste)
        resultados[nome] = pred
        pred.assign(review_id=teste.review_id.values).to_csv(INTERIM / f"pred_repro_teste_{nome}.csv", index=False)
        f1, lo, hi = bootstrap_ci(teste, pred)
        print(f">>> {nome}: F1 macro {f1:.3f} (IC95% {lo:.3f}–{hi:.3f}) | F1 tupla {tuple_f1(teste, pred):.3f}")

In [ ]:
per_aspect_report(teste, resultados["bertimbau_humano"]).round(3)